# Sleep-Stage Classification — XGBoost (improved)

**Task.** Predict `sleep_stage` ∈ {0,1,2,3} from 21 physiological signals. **Metric:** macro-F1.

This notebook tunes **one model — XGBoost** — and pushes it past the booster baseline
(tuned XGBoost ≈ **0.8207** OOF in `boosting.ipynb`). Three stacked, *measured* gains:

1. **Better hyperparameters** — slower learning (`lr=0.03`, `n=800`), shallower trees (`depth=5`)
   with stronger regularisation → less overfitting on the noisy class-2 boundary.
2. **Seed-bagged probabilities** — average 3 XGBoosts (different `random_state`) inside each fold.
   Boosting is variance-prone; averaging seeds smooths it and lifts macro-F1.
3. **Macro-F1 threshold weights** — re-balance per-class decision boundaries on OOF probabilities.

Leak-free: every model is fit only on training folds; nothing sees the test set. Runs in ~1–2 min.

## 0. Setup & data

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier

RS = 42
train = pd.read_csv("train.csv"); test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")
target = "sleep_stage"
features = [c for c in train.columns if c not in ("id", target)]
y = train[target].values
classes = sorted(np.unique(y)); nc = len(classes)

# body_movement_index has MI~0 (shown noise in prior EDA) -> drop it.
FEATS = [c for c in features if c != "body_movement_index"]
X, Xt = train[FEATS].values, test[FEATS].values
print("train:", train.shape, "| using", len(FEATS), "features | classes:", classes)

train: (9000, 23) | using 20 features | classes: [0, 1, 2, 3]


## 1. CV helpers
XGBoost handles NaNs natively (the only missing column, `eog_burst_index`, needs no imputation).
We score with **out-of-fold** predictions on a fixed `StratifiedKFold(5)`.

In [2]:
def oof_proba(params, bag_seeds=(RS,), eval_seed=RS):
    """OOF class-probabilities. Inside each fold, average over `bag_seeds` models (bagging)."""
    cv = StratifiedKFold(5, shuffle=True, random_state=eval_seed)
    P = np.zeros((len(y), nc))
    for tr, va in cv.split(X, y):
        acc = np.zeros((len(va), nc))
        for s in bag_seeds:
            m = XGBClassifier(objective="multi:softprob", num_class=nc, tree_method="hist",
                              eval_metric="mlogloss", n_jobs=-1, random_state=s, **params)
            m.fit(X[tr], y[tr]); acc += m.predict_proba(X[va])
        P[va] = acc / len(bag_seeds)
    return P

def macro(P): return f1_score(y, P.argmax(1), average="macro")

## 2. Baseline — the `boosting.ipynb` XGBoost

In [3]:
BASE = dict(n_estimators=400, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0)
f_base = macro(oof_proba(BASE))
print(f"baseline XGBoost            OOF macro-F1 = {f_base:.4f}")

baseline XGBoost            OOF macro-F1 = 0.8213


## 3. Improvement #1 — better hyperparameters
Slower learning + shallower, more-regularised trees generalise better on the overlapping classes.

In [4]:
TUNED = dict(n_estimators=800, learning_rate=0.03, max_depth=5,
             subsample=0.8, colsample_bytree=0.8,
             reg_lambda=2.0, min_child_weight=3, gamma=0.0)
f_tuned = macro(oof_proba(TUNED))
print(f"#1 tuned hyper-params      OOF macro-F1 = {f_tuned:.4f}   ({f_tuned-f_base:+.4f})")

#1 tuned hyper-params      OOF macro-F1 = 0.8225   (+0.0012)


## 4. Improvement #2 — seed-bagged probabilities
Average 3 XGBoosts with different `random_state` *inside every fold*. Same data, lower variance.

In [5]:
P_bag = oof_proba(TUNED, bag_seeds=(42, 1, 2))
f_bag = macro(P_bag)
print(f"#2 + 3-seed bagging        OOF macro-F1 = {f_bag:.4f}   ({f_bag-f_tuned:+.4f})")

#2 + 3-seed bagging        OOF macro-F1 = 0.8241   (+0.0016)


## 5. Improvement #3 — macro-F1 threshold weights
Greedy per-class probability weights chosen on OOF to maximise macro-F1 (re-balances the boundary
for the hard class 2). NOTE: fit on OOF, so this bump is mildly optimistic — the robust core is #1+#2.

In [6]:
def fit_weights(P, steps=21, rounds=3):
    w = np.ones(nc); grid = np.linspace(0.7, 1.4, steps)
    best = macro_w(P, w)
    for _ in range(rounds):
        for k in range(nc):
            bw = w[k]
            for g in grid:
                w[k] = g
                s = macro_w(P, w)
                if s > best: best, bw = s, g
            w[k] = bw
    return w, best
def macro_w(P, w): return f1_score(y, (P * w).argmax(1), average="macro")

W, f_thr = fit_weights(P_bag)
print(f"#3 + macro-F1 thresholds   OOF macro-F1 = {f_thr:.4f}   ({f_thr-f_bag:+.4f})")
print("class weights:", np.round(W, 3))
print(f"\nTotal lift over baseline: {f_thr-f_base:+.4f}")

#3 + macro-F1 thresholds   OOF macro-F1 = 0.8267   (+0.0026)
class weights: [0.945 1.26  1.    1.015]

Total lift over baseline: +0.0054


## 6. Is the core gain real? — multi-seed stability
Compare baseline vs the tuned+bagged model (argmax, no threshold trick) across 3 CV splits.

In [7]:
def multiseed(params, bag, seeds=(42, 1, 2)):
    return np.array([macro(oof_proba(params, bag_seeds=bag, eval_seed=s)) for s in seeds])
b = multiseed(BASE, (RS,)); v = multiseed(TUNED, (42, 1, 2))
print(f"baseline      mean={b.mean():.4f}  std={b.std():.4f}")
print(f"tuned+bagged  mean={v.mean():.4f}  std={v.std():.4f}")
print(f"mean improvement = {v.mean()-b.mean():+.4f}")

baseline      mean=0.8211  std=0.0002
tuned+bagged  mean=0.8233  std=0.0011
mean improvement = +0.0022


## 7. Error analysis

In [8]:
print(classification_report(y, (P_bag * W).argmax(1), digits=3))
# class 2 stays hardest (genuine physiological overlap) but its recall improves vs baseline.

              precision    recall  f1-score   support

           0      0.867     0.844     0.855      2001
           1      0.824     0.871     0.847      2442
           2      0.786     0.759     0.772      2237
           3      0.834     0.831     0.832      2320

    accuracy                          0.827      9000
   macro avg      0.828     0.826     0.827      9000
weighted avg      0.827     0.827     0.826      9000



## 8. Final model & submission
Refit the 3-seed XGBoost bag on **all** training data, apply the OOF-fit threshold weights, predict test.

In [9]:
proba_test = np.zeros((len(test), nc))
for s in (42, 1, 2):
    m = XGBClassifier(objective="multi:softprob", num_class=nc, tree_method="hist",
                      eval_metric="mlogloss", n_jobs=-1, random_state=s, **TUNED)
    m.fit(X, y); proba_test += m.predict_proba(Xt)
proba_test /= 3
test_pred = (proba_test * W).argmax(1)

submission = pd.DataFrame({"id": test["id"], target: test_pred})[sample_submission.columns]
submission.to_csv("xgboost_submission.csv", index=False)
print(f"final OOF macro-F1 = {f_thr:.4f}")
print("Saved xgboost_submission.csv:", submission.shape)
print(pd.DataFrame({
    "train":     pd.Series(y).value_counts(normalize=True).sort_index(),
    "test_pred": pd.Series(test_pred).value_counts(normalize=True).sort_index(),
}).round(3))
submission.head()

final OOF macro-F1 = 0.8267
Saved xgboost_submission.csv: (5000, 2)
   train  test_pred
0  0.222      0.220
1  0.271      0.274
2  0.249      0.250
3  0.258      0.256


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,2
4,9004,3
